Load the tokenizer:

In [ ]:
from transformers import AutoTokenizer

model_name = "facebook/nllb-200-3.3B"
tokenizer = AutoTokenizer.from_pretrained(model_name, tgt_lang=None)

Prepare your training and validation data:

In [ ]:
rename_dict = {"ja": "jpn_Jpan",
               "en": "eng_Latn",
               "zh-CN": "zho_Hans",
               "th": "tha_Thai",
               "ru": "rus_Cyrl",
               "zh-TW": "zho_Hant",
               "id": "ind_Latn",
               "ko": "kor_Hang",
               "pt": "por_Latn", }

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("all_files_merged.csv")
df.drop("source", axis=1, inplace=True)  # delete the "source" column
df.replace(np.nan, None, inplace=True)  # replace np.nan with None
df.rename(columns=rename_dict, inplace=True)  # rename the columns
df

In [ ]:
language_list = df.columns.tolist()
# x: source language, y: target language
# language_pairs = [[x, y] for index, x in enumerate(language_list) for y in language_list[index:] if x != y]
language_pairs = [["zho_Hans", "kor_Hang"]]
language_pairs

In [ ]:
df_sub_list = []
for pair in language_pairs:
    df_sub = df[pair]
    df_sub = df_sub.dropna(how='any')  # clear all rows include nan
    df_sub = df_sub.drop_duplicates()  # delete exactly the same rows

    # filter the empty language pairs
    if len(df_sub) > 0:
        df_sub_list.append(df_sub)
len(df_sub_list)

find the max length of the encodings

In [ ]:
import sentencepiece as spm
from tqdm.notebook import tqdm

train_encodings_list = []
for df_sub in tqdm(df_sub_list):
    languages = df_sub.columns.tolist()
    language_x, language_y = languages[0], languages[1]
    x_train = df_sub[language_x].to_list()
    y_train = df_sub[language_y].to_list()

    # add basic chars not in the vocabulary of the tokenizer
    chars = list(set(''.join(x_train + y_train)))
    chars_not_in_vocab = [char for char in chars if 3 in tokenizer(char).input_ids]  # 3 is the value of unknown words
    tokenizer.add_tokens(chars_not_in_vocab)

    # use SentencePiece to get vocabulary and add into the tokenizer
    texts_filename = "{0}_{1}_texts.txt".format(language_x, language_y)
    with open(texts_filename, "w", encoding="utf-8") as file:
        file.writelines([text + "\n" for text in x_train + y_train])

    model_prefix = '{0}.{1}.sentencepiece.bpe'.format(language_x, language_y)
    vocab_size = 4200
    spm.SentencePieceTrainer.train(input=texts_filename, model_prefix=model_prefix, vocab_size=vocab_size)

    with open("{0}.vocab".format(model_prefix), "r", encoding="utf-8") as file:
        tokens = [token.strip().split("\t")[0] for token in file.readlines()]

    tokenizer.add_tokens(tokens)

    # Note: we're now creating separate encodings for the inputs and outputs.
    # truncation: truncate the sequence to a shorter length, because sometimes a sequence may be too long for a model to handle
    # padding: Padding is a strategy for ensuring tensors are rectangular by adding a special padding token to shorter sentences.
    #     True or 'longest': Pad to the longest sequence in the batch (or no padding if only a single sequence if provided).
    #     'max_length': Pad to a maximum length specified with the argument max_length or to the maximum acceptable input length for the model if that argument is not provided.
    #     False or 'do_not_pad' (default): No padding (i.e., can output a batch with sequences of different lengths).
    # return_tensors: If set 'pt', will return tensors instead of list of python integers. Acceptable values are PyTorch torch.Tensor objects.
    # max_length (int, optional): Controls the maximum length to use by one of the truncation/padding parameters.
    # 注意：一定要注意这个max_length的使用，当不同的批次要堆叠在一起时，不可以设置为True，而是应该设置为‘max_length'，这样它才能被填充/截断到同一个长度
    tokenizer.src_lang = language_x
    tokenizer.tgt_lang = language_y
    print("tokenizing - source:{}, target:{}".format(language_x, language_y))
    train_encodings = tokenizer(x_train, text_target=y_train, truncation=True, padding="max_length",
                                return_tensors="pt")

    train_encodings_list.append(train_encodings)
len(train_encodings_list)

Convert your encodings into torch Datasets object:

In [ ]:
import torch


class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, data_encoded_list: list):
        self.input_ids = []
        self.attention_mask = []
        self.labels = []
        for data_encoded in data_encoded_list:
            self.input_ids.extend(data_encoded.data["input_ids"])
            self.attention_mask.extend(data_encoded.data["attention_mask"])
            self.labels.extend(data_encoded.data["labels"])

    def __getitem__(self, index):
        item = {"input_ids": self.input_ids[index],
                "attention_mask": self.attention_mask[index],
                "labels": self.labels[index]}
        return item

    def __len__(self):
        return len(self.input_ids)


train_dataset = TranslationDataset(train_encodings_list)

Load the pretrained model:

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
# 调整embedding层的大小
model.resize_token_embeddings(len(tokenizer))

Define your training arguments and train the model:

In [ ]:
custom_model_name = "autodl-tmp/nllb-200-3.3B/zh2ko"

In [ ]:
from transformers import Trainer, TrainingArguments, IntervalStrategy

# fp16：半精度运算，启用后提高一倍以上运算速度，不影响loss
# gradient_accumulation_steps：steps越大，速度越快，loss越高
# gradient_checkpointing：启用后，降低30%左右速度，节省显存2/3
# per_device_train_batch_size：size越大，GPU占用率越大，速度越快，loss越高，几乎成正比
training_args = TrainingArguments(custom_model_name,
                                  num_train_epochs=1,
                                  per_device_eval_batch_size=1,
                                  per_device_train_batch_size=1,
                                  gradient_accumulation_steps=1,
                                  gradient_checkpointing=True,
                                  fp16=True,
                                  logging_strategy=IntervalStrategy.STEPS,
                                  logging_steps=5000,
                                  save_strategy=IntervalStrategy.STEPS,
                                  save_steps=5000,
                                  save_total_limit=1,
                                  )
from torch.utils import checkpoint  #未知的bug：不会自动加载这个包

In [ ]:
trainer = Trainer(
    model=model,  # the instantiated 🤗 Transformers model to be trained
    args=training_args,  # training arguments, defined above
    train_dataset=train_dataset,  # training dataset
)

trainer.train(resume_from_checkpoint=False)

Save your fine-tuned model and tokenizer:

In [ ]:
trainer.save_model(custom_model_name)
tokenizer.save_pretrained(custom_model_name)